In [3]:
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# Define your directories
train_dir = 'D:/CNNProj/archive/Alzheimer_s Dataset/train'
test_dir = 'D:/CNNProj/archive/Alzheimer_s Dataset/test'

# Define image size
IMAGE_SIZE = (128, 128)  # Adjust as needed

def load_and_preprocess_images(directory, image_size):
    images = []
    labels = []
    
    # Iterate through subfolders (categories)
    for label in os.listdir(directory):
        label_path = os.path.join(directory, label)
        
        if os.path.isdir(label_path):
            for filename in os.listdir(label_path):
                file_path = os.path.join(label_path, filename)
                
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    # Read and preprocess image
                    img = cv2.imread(file_path)
                    if img is None:
                        print(f"Warning: Unable to read image at {file_path}")
                        continue
                    
                    img = cv2.resize(img, image_size)
                    img = img / 255.0  # Normalize pixel values
                    
                    images.append(img)
                    labels.append(label)
    
    images = np.array(images)
    labels = np.array(labels)
    
    print(f"Loaded {len(images)} images from {directory}")
    return images, labels

# Load and preprocess training data
train_images, train_labels = load_and_preprocess_images(train_dir, IMAGE_SIZE)

# Load and preprocess testing data
test_images, test_labels = load_and_preprocess_images(test_dir, IMAGE_SIZE)

# Convert labels to one-hot encoding
label_names = sorted(set(train_labels))  # Get unique label names
label_map = {name: i for i, name in enumerate(label_names)}
num_classes = len(label_map)  # Get the number of output classes

train_labels_encoded = np.array([label_map[label] for label in train_labels])
test_labels_encoded = np.array([label_map[label] for label in test_labels])

# One-hot encode labels
train_labels_one_hot = tf.keras.utils.to_categorical(train_labels_encoded, num_classes=num_classes)
test_labels_one_hot = tf.keras.utils.to_categorical(test_labels_encoded, num_classes=num_classes)

# Split training data into training and validation sets
train_images, val_images, train_labels_one_hot, val_labels_one_hot = train_test_split(
    train_images, train_labels_one_hot, test_size=0.2, random_state=42
)

# Print data shapes
print(f'Training images shape: {train_images.shape}')
print(f'Validation images shape: {val_images.shape}')
print(f'Test images shape: {test_images.shape}')
print(f'Training labels shape: {train_labels_one_hot.shape}')
print(f'Validation labels shape: {val_labels_one_hot.shape}')
print(f'Test labels shape: {test_labels_one_hot.shape}')

# Define model architecture
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(num_classes, activation='softmax')  # Output layer using num_classes
])

# Compile model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Print model summary
model.summary()


Loaded 5121 images from D:/CNNProj/archive/Alzheimer_s Dataset/train
Loaded 1279 images from D:/CNNProj/archive/Alzheimer_s Dataset/test
Training images shape: (4096, 128, 128, 3)
Validation images shape: (1025, 128, 128, 3)
Test images shape: (1279, 128, 128, 3)
Training labels shape: (4096, 4)
Validation labels shape: (1025, 4)
Test labels shape: (1279, 4)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)                    │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 126, 126, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 61, 61, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_8 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_8                │ (None, 28, 28, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_8 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │       6,422,784 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,550,340 (24.99 MB)

 Trainable params: 6,549,892 (24.99 MB)

 Non-trainable params: 448 (1.75 KB)

In [4]:
# Train the model
history = model.fit(
    train_images, train_labels_one_hot,
    validation_data=(val_images, val_labels_one_hot),
    epochs=20,  # You can change the number of epochs
    batch_size=32,  # Adjust batch size as needed
    verbose=1
)

# Save the trained model
model.save('alzheimer_cnn_model.h5')


Epoch 1/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 95s 663ms/step - accuracy: 0.4037 - loss: 6.5075 - val_accuracy: 0.2049 - val_loss: 1.3560
Epoch 2/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 80s 626ms/step - accuracy: 0.5038 - loss: 1.1672 - val_accuracy: 0.4868 - val_loss: 1.1331
Epoch 3/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 77s 600ms/step - accuracy: 0.5014 - loss: 1.0891 - val_accuracy: 0.4868 - val_loss: 1.0960
Epoch 4/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 78s 611ms/step - accuracy: 0.4947 - loss: 1.0698 - val_accuracy: 0.4868 - val_loss: 1.0845
Epoch 5/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 77s 601ms/step - accuracy: 0.5214 - loss: 1.0361 - val_accuracy: 0.4868 - val_loss: 1.0788
Epoch 6/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 77s 599ms/step - accuracy: 0.5062 - loss: 1.0380 - val_accuracy: 0.4868 - val_loss: 1.0757
Epoch 7/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 77s 602ms/step - accuracy: 0.5070 - loss: 1.0368 - val_accuracy: 0.4868 - val_loss: 1.0743
Epoch 8/20
128/128 ━━━━━━━━━━━━━━━━━━━━ 80s 627ms/step - accuracy: 0.4912 - loss: 1

In [13]:
!pip install sentencepiece

   ---------------------------------------- 0.0/991.5 kB ? eta -:--:--
   --------------------------------------- 991.5/991.5 kB 23.5 MB/s eta 0:00:00


In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load CNN Model
cnn_model = tf.keras.models.load_model("alzheimer_cnn_model.keras")

# Load NLP Model (T5 for text generation)
tokenizer = T5Tokenizer.from_pretrained("t5-small")
nlp_model = T5ForConditionalGeneration.from_pretrained("t5-small")

# CNN Image Processing
def preprocess_image(img_path):
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"File not found: {img_path}")
    
    img = image.load_img(img_path, target_size=(128, 128))  # Adjust based on your CNN model
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0  # Normalize if required
    return img_array

# Predict Alzheimer's Category
def predict_mri(img_path):
    img_array = preprocess_image(img_path)
    prediction = cnn_model.predict(img_array)
    class_labels = ["Non-Demented", "Very Mild Demented", "Mild Demented", "Moderate Demented"]
    return class_labels[np.argmax(prediction)]

# Generate Medical Summary Report (NLP)
def generate_medical_report(mri_result, patient_details):
    prompt = f"""
    Patient has been diagnosed as {mri_result} based on MRI analysis. 
    Additional medical history includes: {patient_details}.
    Generate a summary report with recommended actions.
    """

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    output = nlp_model.generate(**inputs, max_length=200)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example Usage
mri_image_path = r"D:\CNNProj\test\Moderate-D.jpg"  # Replace with actual path
patient_medical_details = """
- Age: 72
- Family history of Alzheimer's
- Difficulty remembering recent events
- MRI scan shows mild atrophy in hippocampus
- No history of strokes or traumatic brain injury
"""

try:
    mri_result = predict_mri(mri_image_path)
    medical_report = generate_medical_report(mri_result, patient_medical_details)
    
    print("\n🧠 CNN Diagnosis:", mri_result)
    print("\n📄 Generated Medical Report:\n", medical_report)

except FileNotFoundError as e:
    print(e)


D:\CNNProj\venv\lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 20 variables whereas the saved optimizer has 38 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step

🧠 CNN Diagnosis: Mild Demented

📄 Generated Medical Report:
 Mild Demented based on MRI analysis. Additional medical history includes: - Age: 72 - Family history of Alzheimer's - Difficulty remembering recent events - MRI scan shows mild atrophy in hippocampus - No history of strokes or traumatic brain injury.
